# T03: Browse and Search Elements

This tutorial explores the element query API — pagination, filtering by source and
data type, detail views, version history, and semantic alias detection.

**Services required**: backend (`http://localhost:8002`, with ingested data from T02)

**Est. time**: 5 min

In [1]:
# Cell 2 — service availability check
import os

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

# Also check there is data to browse
_check = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/", headers=HEADERS, params={"limit": 1}, timeout=5.0
)
if _check.status_code == 200 and _check.json().get("total", 0) == 0:
    import pytest

    pytest.skip("No elements found — run T02 (02_ingest_schemas.ipynb) first")

✓ Backend available at http://localhost:8002


## 1. Paginate Elements

The elements endpoint supports `limit` and `offset` for pagination.
The response includes a `total` count of all matching elements.

In [2]:
# Page 1
resp1 = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/",
    headers=HEADERS,
    params={"limit": 10, "offset": 0},
    timeout=5.0,
)
assert resp1.status_code == 200
page1 = resp1.json()
assert len(page1["items"]) <= 10

# Page 2
resp2 = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/",
    headers=HEADERS,
    params={"limit": 10, "offset": 10},
    timeout=5.0,
)
assert resp2.status_code == 200
page2 = resp2.json()

total = page1["total"]
print(f"Total elements: {total}")
print(f"Page 1: {len(page1['items'])} items")
print(f"Page 2: {len(page2['items'])} items")

# Store an element ID for later cells
element_id = page1["items"][0]["id"]
print(f"\nFirst element ID: {element_id}")

Total elements: 1417
Page 1: 10 items
Page 2: 10 items

First element ID: 69ab28a5-7369-47c8-8758-1d0fc940fbff


## 2. Filter by Source

Use `source_name` to narrow results to a specific schema source.

In [3]:
# Get available source names
sources_resp = httpx.get(f"{BACKEND_URL}/api/v1/sources/", headers=HEADERS, timeout=5.0)
assert sources_resp.status_code == 200
sources = sources_resp.json()["items"]
source_names = [s["name"] for s in sources]
print(f"Available sources: {source_names}")

if source_names:
    src = source_names[0]
    resp = httpx.get(
        f"{BACKEND_URL}/api/v1/elements/",
        headers=HEADERS,
        params={"source_name": src, "limit": 5},
        timeout=5.0,
    )
    assert resp.status_code == 200
    data = resp.json()
    print(f"\nElements from '{src}': {data['total']} total")
    for item in data["items"]:
        print(f"  - {item['name']} ({item['data_type']})")

    if len(source_names) > 1:
        src2 = source_names[1]
        resp2 = httpx.get(
            f"{BACKEND_URL}/api/v1/elements/",
            headers=HEADERS,
            params={"source_name": src2, "limit": 5},
            timeout=5.0,
        )
        data2 = resp2.json()
        print(f"\nElements from '{src2}': {data2['total']} total")

Available sources: ['BIDS', 'DANDI', 'QS005DbgSrc1773235766', 'QS005Perf1773236575', 'QS005Perf1773236867', 'QS005Src1773235717', 'QS005Src1773235974', 'QS005Src-fix', 'undata']

Elements from 'BIDS': 1417 total
  - url (string)
  - relation (string)
  - identifier (string)
  - name (string)
  - id (string)

Elements from 'DANDI': 1417 total


## 3. Element Detail View

Fetch the full details of a single element by ID.

In [4]:
response = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/{element_id}",
    headers=HEADERS,
    timeout=5.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
element = response.json()
print(f"Element: {element['name']}")
print(f"  ID:              {element['id']}")
print(f"  data_type:       {element['data_type']}")
print(f"  description:     {element.get('description', '(none)')[:80]}")
print(f"  required:        {element.get('required', False)}")
print(f"  source_local_id: {element.get('source_local_id', '?')}")
constraints = element.get("constraints", {})
if constraints:
    print(f"  constraints:     {constraints}")

Element: schemaKey
  ID:              69ab28a5-7369-47c8-8758-1d0fc940fbff
  data_type:       string
  description:     Schema Key
  required:        True
  source_local_id: Resource.schemaKey


## 4. Version History

Every element maintains a full audit history. The `/history` endpoint returns
all previous versions with timestamps and change summaries.

In [5]:
response = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/{element_id}/history",
    headers=HEADERS,
    timeout=5.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
history = response.json()
versions = history if isinstance(history, list) else history.get("items", [])
print(f"Versions: {len(versions)}")
for i, v in enumerate(versions[:3]):
    print(f"  v{i + 1}: {v}")

Versions: 1
  v1: {'id': 'ed7dc5bf-1bba-440c-b0ae-7f2cfccba404', 'element_id': '69ab28a5-7369-47c8-8758-1d0fc940fbff', 'version_num': 1, 'name': 'schemaKey', 'data_type': 'string', 'description': 'Schema Key', 'required': True, 'multivalued': False, 'allowed_values': None, 'constraints': None, 'semantic_graph': None, 'unit': None, 'created_at': '2026-03-12T23:02:07.654621Z', 'created_by_display_name': 'QS005 Test User'}


## 5. Detect Alias Candidates

The `POST /api/v1/aliases/detect` endpoint uses semantic similarity to find
element pairs that might be synonyms across sources. Results above a threshold
are candidates — a human curator must approve them before they become official aliases.

In [6]:
response = httpx.post(
    f"{BACKEND_URL}/api/v1/aliases/detect",
    headers=HEADERS,
    json={"threshold": 0.85, "limit": 5},
    timeout=10.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
candidates = response.json()
pairs = candidates if isinstance(candidates, list) else candidates.get("items", [])
print(f"Alias candidates found: {len(pairs)}")
for pair in pairs[:3]:
    score = pair.get("similarity", pair.get("score", "?"))
    a = pair.get("element_a", {}).get("name", pair.get("element_a_id", "?"))
    b = pair.get("element_b", {}).get("name", pair.get("element_b_id", "?"))
    print(f"  {a!r} ↔ {b!r} (similarity={score})")
print("Note: These are candidates — not yet approved aliases.")

Alias candidates found: 5
  'schemaKey' ↔ 'schemaKey' (similarity=?)
  'schemaKey' ↔ 'schemaKey' (similarity=?)
  'schemaKey' ↔ 'schemaKey' (similarity=?)
Note: These are candidates — not yet approved aliases.


## Next Steps

You've explored pagination, filtering, detail views, version history, and alias
candidate detection.

Next: **[T04: Schema Classes and Element Mappings](04_mappings_aliases.ipynb)** —
create cross-source mappings and curated alias groups.